In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

In [2]:
df = pd.read_excel("sarcastic-classifier.xlsx")
df

,image_name,caption_text,image_text,is_sarcastic,sarcasm_type
0,015.jpg,When your boyfriend falls asleep too early,"""WHY ARE YOU SLEEPING!?""",0,несаркастичний
1,016.jpg,When you commute to the office to do your job ...,Are we in hell?,1,гіперболічний
2,017.jpg,Me after having an emotional breakdown,I'm backkk!!,1,сентиментально-невідповідний
3,018.jpg,Coworker: This is the first time I've seen you...,I'm about to quit,1,сентиментально-невідповідний
4,019.jpg,when someone tells me they're kinda scared of me,That's very nice to hear,1,сентиментально-невідповідний
...,...,...,...,...,...
181,196.jpg,When people come to my funny posts with those ...,"""We don't do that here""",1,формульний
182,197.jpg,*feels a cool breeze,"FALL, IS THAT YOU PLAYA?",1,гіперболічний
183,198.jpg,My brain when it detects the slightest hormone...,I am going to create an environment that is so...,1,гіперболічний
184,199.jpg,All girls know the struggle when there's wind ...,I can't see anything,0,несаркастичний


In [3]:
df["is_sarcastic"].value_counts(normalize=True)

is_sarcastic
1    0.516129
0    0.483871
Name: proportion, dtype: float64

In [4]:
df = df.drop(columns=["image_name", "sarcasm_type"])

In [5]:
df["caption_upper_percentage"] = df["caption_text"].str.findall(r'[A-Z]').str.len() / df["caption_text"].str.findall(r'[a-zA-Z]').str.join('').str.len()
df["image_text_upper_percentage"] = df["image_text"].str.findall(r'[A-Z]').str.len() / df["image_text"].str.findall(r'[a-zA-Z]').str.join('').str.len()
df["caption_upper_percentage"] = df["caption_upper_percentage"].fillna(0)
df["image_text_upper_percentage"] = df["image_text_upper_percentage"].fillna(0)
df["image_text_len"] = df["image_text"].str.len()
df["caption_text_len"] = df["caption_text"].str.len()

In [6]:
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Формуємо повний текст (підпис + текст на зображенні)
df["full_text"] = (
    df["caption_text"].fillna("").astype(str)
    + " "
    + df["image_text"].fillna("").astype(str)
)

# 2. Визначаємо числові ознаки та цільову змінну
numerical_cols = [
    "caption_upper_percentage",
    "image_text_upper_percentage",
]

X_text = df["full_text"]
X_num = df[numerical_cols]
y = df["is_sarcastic"]

# 3. Розбиття на train / test (запобігає data leakage для TF-IDF та Scaler)
X_train_text, X_test_text, X_train_num, X_test_num, y_train, y_test = (
    train_test_split(X_text, X_num, y, test_size=0.2, random_state=42)
)

# 4. TF-IDF векторизація тексту (уніграми + біграми)
tfidf = TfidfVectorizer(
    max_features=300,  # оптимально для невеликого датасету (~186 рядків)
    stop_words="english",
    ngram_range=(1, 2),  # враховує стійкі фрази (наприклад, "yeah right")
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# 5. Масштабування числових ознак
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)

# 6. Об'єднання розрідженої TF-IDF матриці та числових ознак
X_train = hstack([X_train_tfidf, csr_matrix(X_train_num_scaled)])
X_test = hstack([X_test_tfidf, csr_matrix(X_test_num_scaled)])

# Зберігаємо імена всіх ознак для подальшого аналізу важливості
all_feature_names = list(tfidf.get_feature_names_out()) + numerical_cols

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (148, 302), X_test shape: (38, 302)


In [18]:
import time
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

# Ініціалізація моделі
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, C=100),
}

# Список метрик, які ми хочемо отримати
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metrics_report = []

for model_name, model in models.items():
    start_time = time.time()

    cv_results = cross_validate(model, X_train, y_train, cv=5, scoring=scoring)
    
    total_time = time.time() - start_time
    
    # Беремо середнє значення метрик по всіх 5 фолдах
    metrics_report.append({
        "Model": model_name,
        "Accuracy": cv_results['test_accuracy'].mean(),
        "Precision (CT)": cv_results['test_precision'].mean(),
        "Recall (CT)": cv_results['test_recall'].mean(),
        "F1 Score": cv_results['test_f1'].mean(),
        "ROC AUC": cv_results['test_roc_auc'].mean(),
        "Time (sec)": round(total_time, 2)
    })

report_df = pd.DataFrame(metrics_report)
report_df = report_df.sort_values(by="ROC AUC", ascending=False).reset_index(drop=True)

print("=== Результати 5-Fold Крос-валідації ===")
print(report_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

=== Результати 5-Fold Крос-валідації ===
              Model  Accuracy  Precision (CT)  Recall (CT)  F1 Score  ROC AUC  Time (sec)
Logistic Regression    0.6825          0.6874       0.6958    0.6842   0.7380      0.1000


In [19]:
models["Logistic Regression"].fit(X_train, y_train)
models["Logistic Regression"].score(X_test, y_test)

0.7105263157894737

In [9]:
model = models["Logistic Regression"]
importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': model.coef_[0],
    'Importance': np.abs(model.coef_[0])
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

In [10]:
importance_df

,Feature,Coefficient,Importance
135,falls,-12.168058,12.168058
134,fall,9.707979,9.707979
141,forgot,-8.378292,8.378292
187,make,-8.355114,8.355114
20,ask,8.142217,8.142217
...,...,...,...
31,brain,0.150011,0.150011
230,scared,0.146517,0.146517
26,best,0.087771,0.087771
83,come,0.064966,0.064966


In [11]:
importance_df.loc[importance_df["Feature"].isin(numerical_cols)]

,Feature,Coefficient,Importance
301,image_text_upper_percentage,1.300136,1.300136
300,caption_upper_percentage,-0.721721,0.721721


In [12]:
import re

def analyze_sarcasm(caption_text, image_text, model, tfidf, scaler):
    caption_str = str(caption_text) if pd.notna(caption_text) else ""
    image_str = str(image_text) if pd.notna(image_text) else ""
    full_text = caption_str + " " + image_str
    
    def get_upper_perc(text):
        alphas = "".join(re.findall(r'[a-zA-Z]', text))
        uppers = re.findall(r'[A-Z]', text)
        return len(uppers) / len(alphas) if len(alphas) > 0 else 0.0
        
    num_features = [[
        get_upper_perc(caption_str), # caption_upper_percentage
        get_upper_perc(image_str),   # image_text_upper_percentage
        len(image_str),              # image_text_len
        len(caption_str)             # caption_text_len
    ]]
    
    X_tfidf = tfidf.transform([full_text])
    X_num = scaler.transform(num_features)
    X_combined = hstack([X_tfidf, csr_matrix(X_num)])
    
    pred_class = model.predict(X_combined)[0]
    probabilities = model.predict_proba(X_combined)[0]
    confidence = probabilities[pred_class]
    
    feature_names = tfidf.get_feature_names_out()
    nonzero_indices = X_tfidf.nonzero()[1]
    
    caption_words_impact = {}
    image_words_impact = {}
    
    caption_lower = caption_str.lower()
    image_lower = image_str.lower()
    
    for idx in nonzero_indices:
        word = feature_names[idx]
        tfidf_val = X_tfidf[0, idx]
        coef = model.coef_[0][idx]
        
        impact = tfidf_val * coef 
        
        if word in caption_lower:
            caption_words_impact[word] = round(impact, 4)
        if word in image_lower:
            image_words_impact[word] = round(impact, 4)
            
    caption_words_impact = dict(sorted(caption_words_impact.items(), key=lambda item: abs(item[1]), reverse=True))
    image_words_impact = dict(sorted(image_words_impact.items(), key=lambda item: abs(item[1]), reverse=True))
    
    result = {
        "prediction": "Сарказм (1)" if pred_class == 1 else "Не сарказм (0)",
        "confidence": round(confidence, 4),
        "caption_words": caption_words_impact,
        "image_words": image_words_impact
    }
    
    return result

In [13]:
caption = "I love my life very much"
image = "Can I die already please"

result = analyze_sarcasm(caption, image, model, tfidf, scaler)

print(f"Результат: {result['prediction']}")
print(f"Впевненість: {result['confidence'] * 100:.2f}%")
print("\n--- Слова з підпису (Caption) ---")
for word, score in result['caption_words'].items():
    direction = "➡️ Сарказм" if score > 0 else "⬅️ Не сарказм"
    print(f"'{word}': {score} ({direction})")

print("\n--- Слова з картинки (Image Text) ---")
for word, score in result['image_words'].items():
    direction = "➡️ Сарказм" if score > 0 else "⬅️ Не сарказм"
    print(f"'{word}': {score} ({direction})")

C:\Users\rodio\PycharmProjects\DataMining\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


ValueError: X has 4 features, but StandardScaler is expecting 2 features as input.